In [1]:
import os, sys, json
import numpy as np
import mdtraj as md
from datetime import datetime

from openmm.app import *
from openmm import *
from openmm.unit import *

In [2]:
def write_structure(sim: Simulation, pdb_fn: str):
    with open(pdb_fn, 'w') as f:
        PDBFile.writeFile(simulation.topology, simulation.context.getState(getPositions=True).getPositions(), f)
    print(f'Wrote: {pdb_fn}')

In [3]:
def get_positions_from_pdb(fname_pdb, fname_prmtop):
    c = md.load(fname_pdb, top=fname_prmtop)
    return c.xyz[0, :, :]

def run_nvt(initial_pdb_fn, prmtop_fn, std_out_fn, dcd_out_fn,
            dt=2.0, temp=300.0, n_total=5000000000, n_dcd=5000, n_stdout=5000):

    #Initialize OpenMM AMBER Prmtop
    prmtop = AmberPrmtopFile(prmtop_fn)
    
    #Create Gas phase system from prmtop
    system = prmtop.createSystem(nonbondedMethod=NoCutoff, constraints=None)
    integrator = LangevinIntegrator(temp*kelvin, 1.0/picosecond, dt*femtoseconds)
    
    #Try to use both GPU platforms and fall back on CPU if neither work
    try: #TRY opencl first
        print('OPENCL')
        platform = Platform.getPlatformByName('OpenCL')
        properties = {'OpenCLPrecision': 'mixed'}
        simulation = Simulation(prmtop.topology, system, integrator, platform, properties)
    except:
        try: #First fallback to CUDA
            print('CUDA')
            platform = Platform.getPlatformByName('CUDA')
            properties = {'CudaPrecision': 'mixed'}
            simulation = Simulation(prmtop.topology, system, integrator, platform, properties)
        except: #Final Fallback to cpu
            print('CPU')
            simulation = Simulation(prmtop.topology, system, integrator)
    
    #Set initial coordinates to the pdb file provided
    simulation.context.setPositions(get_positions_from_pdb(initial_pdb_fn, prmtop_fn))# nm to nm
    
    #Set a random set of initial velocities based on temperature
    simulation.context.setVelocitiesToTemperature(temp*kelvin)
    
    #Report status of simulation to text file (std_out_fn) every (n_stdout) steps
    SDR = StateDataReporter(std_out_fn, n_stdout, step=True, time=True,
                            potentialEnergy=True, temperature=True, remainingTime=True,
                            totalSteps=n_total, separator='   ::   ')
    simulation.reporters.append(SDR)
    
    #Write the coordinates to DCD file (dcd_out_fn) every (n_dcd) steps (default 10ps)
    DCR = DCDReporter(dcd_out_fn, n_dcd)
    simulation.reporters.append(DCR)
    
    #Run the simulation
    simulation.step(n_total)        

def run_npt(pdb_in, system_xml_in, fn_stdout, fn_dcd,
            temp=300, press=1, nstdout=5000, ndcd=5000, nsteps=500000000, dt=2):
    
    start = datetime.now()
    with open(system_xml_in) as f:
        system = XmlSerializer.deserialize(f.read())
    system.addForce(MonteCarloBarostat(press*bar, temp*kelvin))
    integrator = LangevinIntegrator(temp*kelvin, 1/picosecond, dt*femtosecond)
    pdb = PDBFile(pdb_in)
    try:
        platform = Platform.getPlatformByName('OpenCL')
        properties = {'OpenCLPrecision': 'mixed'}
        simulation = Simulation(pdb.topology, system, integrator, platform, properties)
    except:
        simulation = Simulation(pdb.topology, system, integrator)
    simulation.context.setPositions(pdb.positions)
    simulation.context.setVelocitiesToTemperature(temp)
    SDR = app.StateDataReporter(fn_stdout, nstdout, step=True, time=True,
                                potentialEnergy=True, temperature=True, progress=False,
                                remainingTime=True, speed=False, volume=True,
                                totalSteps=nsteps, separator=' : ')
    simulation.reporters.append(SDR)
    DCDR = app.DCDReporter(fn_dcd, ndcd)
    simulation.reporters.append(DCDR)
    print(f"Prep Done after {datetime.now() - start}")
    simulation.minimizeEnergy()
    print(f"Minimize Done after {datetime.now() - start}")
    print(f'Starting Simulation with forces {simulation.system.getForces()}')
    print(f'Starting Simulation with box_vectors {simulation.system.getDefaultPeriodicBoxVectors()}')
    simulation.step(nsteps)
    print(f"Simulation Done after {datetime.now() - start}")

In [ ]:
initial_pdb_fn = 'Simulation/3mxf_complex_final.pdb'
system_xml_fn = 'Simulation/3mxf_complex_final.xml'
std_out_fn = 'Simulation/3xmf_100ns.stdout'
dcd_out_fn = 'Simulation/3mxf_100ns.dcd'

run_npt(initial_pdb_fn, system_xml_fn, std_out_fn, dcd_out_fn)

1 warning generated.


Prep Done after 0:00:05.413483
Minimize Done after 0:00:06.094614
Starting Simulation with forces [<openmm.openmm.NonbondedForce; proxy of <Swig Object of type 'OpenMM::NonbondedForce *' at 0x7f0227c7d8c0> >, <openmm.openmm.PeriodicTorsionForce; proxy of <Swig Object of type 'OpenMM::PeriodicTorsionForce *' at 0x7f0227c7dce0> >, <openmm.openmm.HarmonicAngleForce; proxy of <Swig Object of type 'OpenMM::HarmonicAngleForce *' at 0x7f0227c7d7d0> >, <openmm.openmm.HarmonicBondForce; proxy of <Swig Object of type 'OpenMM::HarmonicBondForce *' at 0x7f0227c7d770> >, <openmm.openmm.MonteCarloBarostat; proxy of <Swig Object of type 'OpenMM::MonteCarloBarostat *' at 0x7f0227c7d710> >]
Starting Simulation with box_vectors [Quantity(value=Vec3(x=6.896100000000001, y=0.0, z=0.0), unit=nanometer), Quantity(value=Vec3(x=0.0, y=6.896100000000001, z=0.0), unit=nanometer), Quantity(value=Vec3(x=0.0, y=0.0, z=6.896100000000001), unit=nanometer)]
